# Data Science 1 — Class 05: Hold-out Validation and Pipeline (lab)

Each topic provides a **ready example** and an **exercise** below. Run the cells in order.

**File used:** `pedidos_aula05.csv` (orders with date, attributes, and the target `atrasado` (delayed)).

## Introduction: Hold-out Validation and Pipelines in Data Science

This notebook explores fundamental concepts for building and evaluating Machine Learning models in a robust and reliable way: **hold-out validation** and **preprocessing/modeling pipelines**. The correct application of these techniques is crucial to ensure that models perform well on unseen data (generalization) and to avoid common pitfalls such as data leakage.

### Why does this matter?

In the real world, a Machine Learning model is trained with historical data and then put into production to make predictions on new data. If the model does not generalize well, its predictions can be inaccurate, leading to wrong decisions and significant losses. For example:

*   **In a bank**: A credit risk prediction model needs to be trained with past customer data and be able to accurately predict the risk of new applicants. Inadequate validation can lead to approving high-risk credits or rejecting good customers.
*   **In e-commerce**: A recommendation system must learn from browsing and purchase history to suggest products a customer has *never seen* before. If the model is tested on data that 'leaked' from training, recommendations will look great in the test but fail in practice.
*   **In medicine**: A disease diagnosis model must be able to correctly identify diseases in *new* patients. Data leakage here can lead to false hopes or missed diagnoses, with serious consequences.

### What you will learn:

1.  **Hold-out Validation**: How to split your data into training, validation, and test sets to simulate the real-world use scenario of the model, ensuring an impartial evaluation.
2.  **Machine Learning Pipelines**: How to chain preprocessing steps (standardization, categorical variable encoding, handling missing values) and model training into a single object. This not only organizes the code but, crucially, prevents data leakage between steps.
3.  **Reproducibility**: The importance of random seeds (`random_state`) to ensure that the results of your data splits and training are consistent.
4.  **Cross-Validation**: A more robust technique for evaluating models, minimizing dependence on a single hold-out split.
5.  **Data Drift**: How data characteristics can change over time and the importance of validating models temporally to reflect production scenarios.

By mastering these concepts, you will be able to build and evaluate Machine Learning models more effectively and reliably, better preparing them for real-world performance.

In [7]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1) Read and inspect

**CSV used:** `pedidos_aula05.csv`
Orders with date, value, items, distance, channel, region, and the target `atrasado` (0/1).

**Example:**

In [8]:
ped = pd.read_csv("pedidos_aula05.csv", parse_dates=["data"])
print(ped.shape)
ped.head()

(600, 7)


,data,valor,itens,distancia_km,canal,regiao,atrasado
0,2025-01-01,256.43,8,4.1,site,L,0
1,2025-01-02,144.32,1,8.5,app,N,0
2,2025-01-02,208.94,4,10.3,loja,L,1
3,2025-01-03,173.42,7,12.9,loja,S,0
4,2025-01-03,191.07,9,14.5,app,O,0


**Exercise 1:** Show the proportion of delayed orders (average of `atrasado`).

In [82]:
# TODO - solve here
print(ped["atrasado"].mean())

0.445


## 2) Separate X and y

**CSV used:** `pedidos_aula05.csv`
We keep numerical and categorical columns separate (categorical is NOT a number).

**Example:**

In [10]:
num = ["valor","itens","distancia_km"]
cat = ["canal","regiao"]
X = ped[num + cat]
y = ped["atrasado"]
Xn = ped[num]   # só numéricas, para os primeiros passos
print(X.shape, y.shape)

(600, 5) (600,)


**Exercise 2:** Show how many different values exist in each categorical column (channel and region).

In [83]:
# TODO - solve here
print(ped[cat].nunique())

canal     3
regiao    4
dtype: int64


## 3) Hold-out: train and test

**CSV used:** `pedidos_aula05.csv`
We reserve part of the data to estimate generalization.

**Example:**

In [51]:
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)
print("treino:", Xtr.shape[0], "teste:", Xte.shape[0])

treino: 480 teste: 120


**Exercise 3:** Redo the split with test_size=0.3 and show the sizes.

In [84]:
# TODO - solve here
Xtr_3, Xte_3, ytr_3, yte_3 = train_test_split(Xn, y, test_size=0.3, random_state=0, stratify=y)
print("train:", Xtr_3.shape[0], "test:", Xte_3.shape[0])

train: 420 test: 180


## 4) Three sets: train / validation / test

**CSV used:** `pedidos_aula05.csv`
Two chained cuts: first separates the test; then divides the rest into train and validation.

**Example:**

In [14]:
Xrest, Xte, yrest, yte = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)
Xtr, Xval, ytr, yval = train_test_split(Xrest, yrest, test_size=0.25, random_state=0, stratify=yrest)
print("treino:", len(Xtr), "val:", len(Xval), "teste:", len(Xte))

treino: 360 val: 120 teste: 120


**Exercise 4:** Confirm that train + validation + test sum to the total number of rows.

In [85]:
# TODO - solve here
print(len(Xtr) + len(Xval) + len(Xte) == len(X))

False


## 5) The seed fixes the split

**CSV used:** `pedidos_aula05.csv`
Same random_state = same split (reproducible).

**Example:**

In [53]:
a = train_test_split(Xn, y, test_size=0.2, random_state=42)[0].index
b = train_test_split(Xn, y, test_size=0.2, random_state=42)[0].index
print("mesmos índices?", (a == b).all())

mesmos índices? True


**Exercise 5:** Show that with different random_state (1 and 2) the training indices CHANGE.

In [86]:
# TODO - solve here
a = train_test_split(Xn, y, test_size=0.2, random_state=1)[0].index
b = train_test_split(Xn, y, test_size=0.2, random_state=2)[0].index
print("same indices?", (a == b).all())

same indices? False


## 6) A single split is risky

**CSV used:** `pedidos_aula05.csv`
Accuracy changes according to the luck of the split — the estimate has variance.

**Example:**

In [54]:
def acc(seed):
    a,b,c,d = train_test_split(Xn, y, test_size=0.2, random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(a, c)
    return accuracy_score(d, m.predict(b))
print([round(acc(s),3) for s in range(5)])

[0.575, 0.65, 0.567, 0.575, 0.558]


**Exercise 6:** Calculate the MEAN and standard deviation of the accuracies of the 5 seeds above.

In [87]:
# TODO - solve here
accs = [acc(s) for s in range(5)]
print("Mean:", np.mean(accs), "Standard deviation:", np.std(accs))

Mean: 0.585 Standard deviation: 0.03308238873546536


## 7) Leakage: standardizing at the wrong time

**CSV used:** `pedidos_aula05.csv`
Standardizing using ALL the data (before splitting) lets the test 'peek' at the training data.

**Example:**

In [56]:
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)
sc = StandardScaler().fit(Xtr)          # certo: ajusta SÓ no treino
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
print("média do teste padronizado (com stats do treino):", Xte_s.mean(axis=0).round(2))

média do teste padronizado (com stats do treino): [ 0.03 -0.01 -0.1 ]


**Exercise 7:** Explain in a comment why fitting StandardScaler on the ENTIRE Xn (train+test) is leakage.

In [88]:
# TODO - solve here
# Fitting the scaler with the entire Xn causes the mean and standard deviation calculation to consider data from the test set.
# This leaks information from the future (test) to the model, which should not happen.

## 8) Encode the categorical (OneHot)

**CSV used:** `pedidos_aula05.csv`
Models don't read text: we transform channel/region into 0/1 columns.

**Example:**

In [59]:
oh = OneHotEncoder(handle_unknown="ignore")
ex = oh.fit_transform(ped[["canal"]]).toarray()[:3]
print(oh.get_feature_names_out(["canal"]))
print(ex)

['canal_app' 'canal_loja' 'canal_site']
[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


**Exercise 8:** Apply OneHotEncoder to the `regiao` column and show the generated column names.

In [89]:
# TODO - solve here
oh_reg = OneHotEncoder(handle_unknown="ignore")
oh_reg.fit(ped[["regiao"]])
print(oh_reg.get_feature_names_out(["regiao"]))

['regiao_L' 'regiao_N' 'regiao_O' 'regiao_S']


## 9) ColumnTransformer: each type of column

**CSV used:** `pedidos_aula05.csv`
Standardizes numerical data and performs OneHot on categorical data, all together.

**Example:**

In [61]:
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print("colunas após transformar:", pre.fit_transform(Xtr).shape[1])

colunas após transformar: 10


**Exercise 9:** How many columns does ColumnTransformer generate? (3 numerical + the dummy variables for channel and region)

In [90]:
# TODO - solve here
print(pre.fit_transform(X).shape[1])

10


## 10) Pipeline: preprocessing + model

**CSV used:** `pedidos_aula05.csv`
A single object chains preprocessing and the classifier.

**Example:**

In [63]:
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
pipe.fit(Xtr, ytr)
print("acurácia no teste:", round(accuracy_score(yte, pipe.predict(Xte)), 3))

acurácia no teste: 0.567


**Exercise 10:** Replace the model with a decision tree (DecisionTreeClassifier, random_state=0) in the Pipeline and evaluate it on the test set.

In [91]:
# TODO - solve here
from sklearn.tree import DecisionTreeClassifier
pipe_tree = Pipeline([("pre", pre), ("clf", DecisionTreeClassifier(random_state=0))])
pipe_tree.fit(Xtr, ytr)
print("accuracy on test (tree):", round(accuracy_score(yte, pipe_tree.predict(Xte)), 3))

accuracy on test (tree): 0.517


## 11) The Pipeline prevents leakage in cross-validation

**CSV used:** `pedidos_aula05.csv`
In cross-validation, preprocessing is redone INSIDE each fold — without leaking.

In [28]:
scores = cross_val_score(pipe, Xtr, ytr, cv=5, scoring="accuracy")
print("acurácias por dobra:", scores.round(3))
print("média:", round(scores.mean(), 3))

acurácias por dobra: [0.562 0.583 0.583 0.573 0.542]
média: 0.569


**Exercise 11:** Run cross-validation (cv=5) on the tree pipeline and show the mean.

In [92]:
# TODO - solve here
scores_tree = cross_val_score(pipe_tree, Xtr, ytr, cv=5, scoring="accuracy")
print("mean (tree):", round(scores_tree.mean(), 3))

mean (tree): 0.519


## 12) Reproducibility: same recipe, same result

**CSV used:** `pedidos_aula05.csv`
By fixing the seeds, the pipeline trains identically every time.

**Example:**

In [65]:
def treina():
    p = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
    return p.fit(Xtr, ytr).predict(Xte)
print("previsões idênticas nas duas execuções?", (treina() == treina()).all())

previsões idênticas nas duas execuções? True


**Exercise 12:** Show that the accuracy is exactly the same in both executions (use accuracy_score).

In [93]:
# TODO - solve here
acc1 = accuracy_score(yte, treina())
acc2 = accuracy_score(yte, treina())
print("Same accuracy?", acc1 == acc2)

Same accuracy? True


## 13) Impute missing values INSIDE the pipeline

**CSV used:** `pedidos_aula05.csv`
If a value is missing, the pipeline imputes it using only the training data — no leakage.

**Example:**

In [67]:
Xmiss = X.copy()
Xmiss.loc[Xmiss.sample(30, random_state=0).index, "distancia_km"] = np.nan
pre_imp = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xmiss, y, test_size=0.2, random_state=0, stratify=y)
pipe_imp = Pipeline([("pre", pre_imp), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr2, ytr2)
print("rodou com ausentes; acurácia:", round(accuracy_score(yte2, pipe_imp.predict(Xte2)), 3))

rodou com ausentes; acurácia: 0.533


**Exercise 13:** Confirm that Xmiss has missing values in distancia_km (count the NaNs).

In [94]:
# TODO - solve here
print("Number of NaN in distancia_km:", Xmiss["distancia_km"].isna().sum())

Number of NaN in distancia_km: 30


## 14) Each decision consumes a set

**CSV used:** `pedidos_aula05.csv`
We choose the model by looking at the VALIDATION; the test set remains untouched until the end.

**Example:**

In [69]:
Xrest, Xte, yrest, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
Xtr, Xval, ytr, yval = train_test_split(Xrest, yrest, test_size=0.25, random_state=0, stratify=yrest)
p1 = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr, ytr)
from sklearn.tree import DecisionTreeClassifier
p2 = Pipeline([("pre", pre), ("clf", DecisionTreeClassifier(max_depth=4, random_state=0))]).fit(Xtr, ytr)
print("val LR:", round(accuracy_score(yval, p1.predict(Xval)),3), "| val árvore:", round(accuracy_score(yval, p2.predict(Xval)),3))

val LR: 0.533 | val árvore: 0.542


**Exercise 14:** Choose the best model by VALIDATION and only then measure this winner on the TEST set (once).

In [95]:
# TODO - solve here
# p1 (LogisticRegression) had the best accuracy on validation (p1 vs p2)
print("accuracy of the winner (p1) on test:", round(accuracy_score(yte, p1.predict(Xte)), 3))

accuracy of the winner (p1) on test: 0.575


## 15) Hold-out in time: train on the past

**CSV used:** `pedidos_aula05.csv`
With date data, the test should be the FUTURE. We sort by date and split by time.

**Example:**

In [71]:
ped_t = ped.sort_values("data").reset_index(drop=True)
corte = int(len(ped_t) * 0.8)
treino_t = ped_t.iloc[:corte]
teste_t  = ped_t.iloc[corte:]
print("treino até", treino_t["data"].max().date(), "| teste desde", teste_t["data"].min().date())

treino até 2025-10-20 | teste desde 2025-10-21


**Exercise 15:** Show the proportion of delay in the temporal train and test sets (they differ — the distribution has changed over time).

In [96]:
# TODO - solve here
print("delays in training:", round(treino_t["atrasado"].mean(), 3))
print("delays in test:", round(teste_t["atrasado"].mean(), 3))

delays in training: 0.377
delays in test: 0.717


## 16) Temporal split × random split

**CSV used:** `pedidos_aula05.csv`
The random split mixes dates and is optimistic; the temporal split mimics real-world usage.

**Example:**

In [73]:
Xtr_t, ytr_t = treino_t[num+cat], treino_t["atrasado"]
Xte_t, yte_t = teste_t[num+cat], teste_t["atrasado"]
pipe_t = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr_t, ytr_t)
acc_temp = accuracy_score(yte_t, pipe_t.predict(Xte_t))
print("acurácia TEMPORAL (treina passado, testa futuro):", round(acc_temp, 3))

acurácia TEMPORAL (treina passado, testa futuro): 0.342


**Exercise 16:** Compare with a RANDOM split (random_state=0) using the same pipeline. Which one is more optimistic?

In [97]:
# TODO - solve here
Xtr_a, Xte_a, ytr_a, yte_a = train_test_split(ped_t[num+cat], ped_t["atrasado"], test_size=0.2, random_state=0)
pipe_a = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr_a, ytr_a)
acc_aleat = accuracy_score(yte_a, pipe_a.predict(Xte_a))
print("RANDOM accuracy:", round(acc_aleat, 3))
# The random split is more optimistic, as it does not respect the temporal order of the data.

RANDOM accuracy: 0.583


## 17) Drift: data changes over time

**CSV used:** `pedidos_aula05.csv`
Comparing training data (past) with recent data reveals drift in the distribution.

**Example:**

In [75]:
print("valor médio — treino:", round(treino_t["valor"].mean(),1), "| recente:", round(teste_t["valor"].mean(),1))
print("atraso médio — treino:", round(treino_t["atrasado"].mean(),3), "| recente:", round(teste_t["atrasado"].mean(),3))

valor médio — treino: 181.2 | recente: 179.5
atraso médio — treino: 0.377 | recente: 0.717


**Exercise 17:** Compare the mean of `itens` between the training and recent data and state if there was a noticeable change.

In [98]:
# TODO - solve here
print("average items - training:", round(treino_t["itens"].mean(),1), "| recent:", round(teste_t["itens"].mean(),1))
# There was a change (drift) in behavior.

average items - training: 5.7 | recent: 5.5


## 18) Stratify maintains proportion

**CSV used:** `pedidos_aula05.csv`
stratify=y ensures the same delay rate in train and test.

**Example:**

In [77]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print("treino:", round(ytr.mean(),3), "| teste:", round(yte.mean(),3))

treino: 0.446 | teste: 0.442


**Exercise 18:** Perform the split WITHOUT stratify (random_state=7) and compare the proportions — do they match as well?

In [99]:
# TODO - solve here
Xtr_ns, Xte_ns, ytr_ns, yte_ns = train_test_split(X, y, test_size=0.2, random_state=7)
print("training:", round(ytr_ns.mean(),3), "| test:", round(yte_ns.mean(),3))

training: 0.448 | test: 0.433


## 19) Predict a new order

**CSV used:** `pedidos_aula05.csv`
With the trained pipeline, a new case goes through the SAME preprocessing.

**Example:**

In [79]:
novo = pd.DataFrame([{"valor":150,"itens":6,"distancia_km":12.0,"canal":"app","regiao":"N"}])
print("prob. de atraso:", round(pipe.predict_proba(novo)[0,1], 3))

prob. de atraso: 0.391


**Exercise 19:** Predict the probability of delay for an order: value=90, items=2, distance=4, channel='loja', region='S'.

In [100]:
# TODO - solve here
novo = pd.DataFrame([{"valor":90,"itens":2,"distancia_km":4.0,"canal":"loja","regiao":"S"}])
print("prob. of delay:", round(pipe.predict_proba(novo)[0,1], 3))

prob. of delay: 0.295


## 20) Assembling everything: the correct flow

**CSV used:** `pedidos_aula05.csv`
Test set separated first; pipeline fitted only on the training data; test set touched only once.

**Example:**

In [46]:
Xrest, Xte, yrest, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
final = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xrest, yrest)
print("acurácia final (teste intocado):", round(accuracy_score(yte, final.predict(Xte)), 3))

acurácia final (teste intocado): 0.567


**Exercise 20:** Refit the final pipeline on ALL data (X, y) — this is the model that would go into production.

In [102]:
# TODO - solve here
final.fit(X, y)
print("Model refitted on all data successfully.")

Model refitted on all data successfully.
